# Dataset v3: partial fine-tuning of the LRW encoder

Контролируемый эксперимент: меняем только способ обучения визуального энкодера.

```text
текущий baseline: frozen LRW 3D Conv + ResNet18 -> BiGRU
новый вариант:     LRW 3D Conv + ResNet18 (trainable layer4) -> BiGRU
```

Используются dataset v3, grayscale, `uniform shuffle`, validation-спикер `spk06` и seed 42. Test split не читается. Сначала запускается один seed: три seed нужны только после улучшения не менее чем на 0.02 validation macro-F1.

## 1. Подключить Google Drive и получить зависимости

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

## 2. Скачать официальный LRW-код и найти dataset v3

In [ ]:
from pathlib import Path
import copy
import random
import shutil
import subprocess
import sys
import time
import zipfile

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
upstream_store = drive_root / 'upstream_lrw_pretrained'
upstream_store.mkdir(parents=True, exist_ok=True)

if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)

checkpoint_path = upstream_store / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    downloaded = gdown.download(
        id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD',
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

extract_dir = Path('/content/lipreading_data_v3')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_landmark_dataset_v3'
    / 'speaker_top10'
)
expected_split_dir = extract_dir / 'ru_dataset' / relative_split_dir
if not expected_split_dir.exists():
    matches = list(Path('/content/drive').rglob(
        'colab_lipreading_dataset_v3_train_val.zip'
    ))
    if not matches:
        raise FileNotFoundError(
            'Upload colab_lipreading_dataset_v3_train_val.zip to Google Drive.'
        )
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0], 'r') as archive:
        archive.extractall(extract_dir)
    print('Dataset restored from:', matches[0])

sys.path.insert(0, str(upstream_dir))
data_root = extract_dir / 'ru_dataset'
split_dir = data_root / relative_split_dir
output_dir = drive_root / 'lrw_layer4_finetune_dataset_v3'
output_dir.mkdir(parents=True, exist_ok=True)
print('dataset:', data_root)
print('checkpoint:', checkpoint_path)
print('output:', output_dir)

## 3. Прочитать только train и validation

Test намеренно не загружается: по нему нельзя подбирать архитектуру или learning rate.

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('Enable a GPU runtime before running this experiment.')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}

assert len(train_df) == 1901
assert len(val_df) == 105
assert set(train_df['speaker_id']).isdisjoint(set(val_df['speaker_id']))
assert all((data_root / path).is_file() for path in train_df['clip_path'])
assert all((data_root / path).is_file() for path in val_df['clip_path'])

print('device:', device)
print('train:', len(train_df), sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), sorted(val_df['speaker_id'].unique()))
print('vocab:', vocab)
print('TEST SPLIT IS NOT LOADED.')

## 4. Датасет видеоклипов

Вход модели: 24 grayscale-кадра, `88x88`, с LRW-нормализацией. Здесь аугментаций нет, чтобы измерить только эффект partial fine-tuning.

In [ ]:
def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frames = []
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        offset = (input_size - crop_size) // 2
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()

    if not frames:
        raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)


class EndToEndVideoDataset(Dataset):
    def __init__(self, metadata, data_root):
        self.metadata = metadata.reset_index(drop=True)
        self.data_root = data_root

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        video = load_lrw_video(self.data_root / row['clip_path'])
        label = word_to_idx[row['word']]
        return video, label


def make_video_loaders(seed, batch_size=8):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        EndToEndVideoDataset(train_df, data_root),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        EndToEndVideoDataset(val_df, data_root),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_loader, val_loader

sample_x, sample_y = next(iter(make_video_loaders(seed=42)))
print('input:', sample_x.shape, sample_x.min().item(), sample_x.max().item())
print('labels:', sample_y.tolist())

## 5. LRW encoder + BiGRU

Сначала загружаются официальные LRW-веса. Затем frozen baseline с `uniform shuffle` и seed 42 точно воспроизводится на сохранённых LRW-признаках. Его лучшие веса BiGRU становятся стартом для partial fine-tuning.

In [ ]:
from lipreading.model import Lipreading


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


class PartialFineTuneLipReader(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, videos):
        features = self.encoder(
            videos, lengths=[videos.shape[2]] * videos.shape[0]
        )
        return self.head(features)


tcn_options = {
    'num_layers': 4,
    'kernel_size': [3, 5, 7],
    'dropout': 0.2,
    'dwpw': False,
    'width_mult': 1,
}
encoder = Lipreading(
    modality='video',
    num_classes=500,
    tcn_options=tcn_options,
    backbone_type='resnet',
    relu_type='swish',
    width_mult=1.0,
    use_boundary=False,
    extract_feats=True,
)
official_checkpoint = load_torch_file(checkpoint_path)
encoder.load_state_dict(
    official_checkpoint.get('model_state_dict', official_checkpoint), strict=True
)
head = BiGRUClassifier(len(vocab))

model = PartialFineTuneLipReader(encoder, head).to(device)
assert hasattr(model.encoder.trunk, 'layer4')
print('Model is ready.')

## 6. Метрики и один train epoch

In [ ]:
def freeze_batchnorm(module):
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        module.eval()


def count_trainable_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def compute_metrics(labels, predictions):
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(
            labels, predictions, average='macro', zero_division=0
        ),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'predicted_classes': len(set(predictions)),
    }


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    labels = []
    predictions = []
    with torch.no_grad():
        for videos, targets in loader:
            videos = videos.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            logits = model(videos)
            total_loss += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = compute_metrics(labels, predictions)
    metrics['loss'] = total_loss / len(labels)
    metrics['labels'] = labels
    metrics['predictions'] = predictions
    return metrics


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    model.encoder.apply(freeze_batchnorm)
    total_loss = 0.0
    labels = []
    predictions = []
    for videos, targets in loader:
        videos = videos.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=True):
            logits = model(videos)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=5.0
        )
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * targets.size(0)
        labels.extend(targets.cpu().tolist())
        predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())
    metrics = compute_metrics(labels, predictions)
    metrics['loss'] = total_loss / len(labels)
    return metrics

## 7. Stage A: воспроизвести frozen baseline

Обучаем только BiGRU на заранее сохранённых frozen LRW-признаках. Это быстро и даёт честную исходную точку с тем же `uniform shuffle`, seed 42 и early stopping, что в sampler ablation.

In [ ]:
class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


def evaluate_head(head, loader, criterion):
    head.eval()
    loss_sum = 0.0
    labels = []
    predictions = []
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = head(features)
            loss_sum += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = compute_metrics(labels, predictions)
    metrics['loss'] = loss_sum / len(labels)
    return metrics


SEED = 42
BATCH_SIZE = 8
criterion = nn.CrossEntropyLoss()
feature_root = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
train_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_features.pt'
val_cache_path = feature_root / 'dataset_v3_val_lrw_resnet18_features.pt'
if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Frozen feature cache is missing. Run the dataset v3 LRW notebook first.'
    )
train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)

set_reproducible_seed(SEED)
model.head = BiGRUClassifier(len(vocab)).to(device)
feature_generator = torch.Generator().manual_seed(SEED)
feature_train_loader = DataLoader(
    FeatureSequenceDataset(train_cache), batch_size=32, shuffle=True,
    generator=feature_generator, num_workers=0
)
feature_val_loader = DataLoader(
    FeatureSequenceDataset(val_cache), batch_size=32, shuffle=False,
    num_workers=0
)

for parameter in model.encoder.parameters():
    parameter.requires_grad = False
for parameter in model.head.parameters():
    parameter.requires_grad = True
baseline_optimizer = torch.optim.AdamW(
    model.head.parameters(), lr=1e-3, weight_decay=1e-4
)
baseline_best_f1 = -1.0
baseline_best_loss = float('inf')
baseline_best_state = None
baseline_without_improvement = 0

for epoch in range(1, 31):
    model.head.train()
    for features, targets in feature_train_loader:
        features = features.to(device)
        targets = targets.to(device)
        baseline_optimizer.zero_grad(set_to_none=True)
        logits = model.head(features)
        loss = criterion(logits, targets)
        loss.backward()
        baseline_optimizer.step()
    baseline_metrics = evaluate_head(model.head, feature_val_loader, criterion)
    improved = (
        baseline_metrics['macro_f1'] > baseline_best_f1
        or (
            baseline_metrics['macro_f1'] == baseline_best_f1
            and baseline_metrics['loss'] < baseline_best_loss
        )
    )
    if improved:
        baseline_best_f1 = baseline_metrics['macro_f1']
        baseline_best_loss = baseline_metrics['loss']
        baseline_best_state = copy.deepcopy(model.head.state_dict())
        baseline_without_improvement = 0
    else:
        baseline_without_improvement += 1
    if baseline_without_improvement >= 7:
        break

model.head.load_state_dict(baseline_best_state)
FROZEN_BASELINE_MACRO_F1 = baseline_best_f1
print('Reproduced frozen baseline macro-F1:', round(baseline_best_f1, 4))
print('Expected approximately: 0.5454')

# From this point onward the model reads videos end to end.
set_reproducible_seed(SEED)
train_loader, val_loader = make_video_loaders(SEED, BATCH_SIZE)
scaler = torch.cuda.amp.GradScaler(enabled=True)
initial_metrics = evaluate(model, val_loader, criterion)
print('Frozen validation from raw videos:', initial_metrics)

# Stage B: only ResNet18.layer4 and the BiGRU head are trainable.
for parameter in model.encoder.trunk.layer4.parameters():
    parameter.requires_grad = True
for module in model.encoder.trunk.layer4.modules():
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        for parameter in module.parameters():
            parameter.requires_grad = False

layer4_parameters = [
    p for p in model.encoder.trunk.layer4.parameters() if p.requires_grad
]
head_parameters = [p for p in model.head.parameters() if p.requires_grad]
print('trainable total:', count_trainable_parameters(model))
print('trainable layer4:', sum(p.numel() for p in layer4_parameters))
print('trainable head:', sum(p.numel() for p in head_parameters))

## 8. Fine-tune seed 42

У `layer4` learning rate в 30 раз меньше, чем у BiGRU. Это позволяет адаптировать признаки к русским словам, не разрушая LRW-pretraining. Ожидаемое время на Colab T4: примерно 25–60 минут.

In [ ]:
MAX_EPOCHS = 15
PATIENCE = 5
REQUIRED_GAIN = 0.02

optimizer = torch.optim.AdamW([
    {'params': layer4_parameters, 'lr': 1e-5},
    {'params': head_parameters, 'lr': 3e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)

best_macro_f1 = -1.0
best_val_loss = float('inf')
best_state = None
best_metrics = None
best_epoch = 0
epochs_without_improvement = 0
history = []
start = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler
    )
    val_metrics = evaluate(model, val_loader, criterion)
    history.append({
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'train_macro_f1': train_metrics['macro_f1'],
        'validation_loss': val_metrics['loss'],
        'validation_accuracy': val_metrics['accuracy'],
        'validation_macro_f1': val_metrics['macro_f1'],
        'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
        'predicted_classes': val_metrics['predicted_classes'],
        'layer4_lr': optimizer.param_groups[0]['lr'],
        'head_lr': optimizer.param_groups[1]['lr'],
    })

    improved = (
        val_metrics['macro_f1'] > best_macro_f1
        or (
            val_metrics['macro_f1'] == best_macro_f1
            and val_metrics['loss'] < best_val_loss
        )
    )
    if improved:
        best_macro_f1 = val_metrics['macro_f1']
        best_val_loss = val_metrics['loss']
        best_epoch = epoch
        best_metrics = copy.deepcopy(val_metrics)
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'epoch {epoch:02d}/{MAX_EPOCHS} | '
        f'train F1={train_metrics["macro_f1"]:.4f} | '
        f'val F1={val_metrics["macro_f1"]:.4f} | '
        f'val acc={val_metrics["accuracy"]:.4f} | '
        f'classes={val_metrics["predicted_classes"]}/10'
    )
    scheduler.step()
    if epochs_without_improvement >= PATIENCE:
        print('Early stopping.')
        break

elapsed = time.perf_counter() - start
history_df = pd.DataFrame(history)
history_df.to_csv(output_dir / 'layer4_finetune_seed42_history.csv', index=False)
torch.save({
    'model_state_dict': best_state,
    'vocab': vocab,
    'seed': SEED,
    'best_epoch': best_epoch,
    'validation_metrics': best_metrics,
    'test_used': False,
}, output_dir / 'layer4_finetune_seed42_best.pt')

gain = best_macro_f1 - FROZEN_BASELINE_MACRO_F1
print('best epoch:', best_epoch)
print('best validation macro-F1:', round(best_macro_f1, 4))
print('frozen seed42 macro-F1:', round(FROZEN_BASELINE_MACRO_F1, 4))
print('gain:', round(gain, 4))
print('time, minutes:', round(elapsed / 60, 1))
print('GATE:', 'PASS' if gain >= REQUIRED_GAIN else 'NO PASS')
print('TEST SPLIT WAS NOT USED.')

## 9. Графики и решение

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['validation_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['train_macro_f1'], label='train')
axes[1].plot(
    history_df['epoch'], history_df['validation_macro_f1'], label='validation'
)
axes[1].axhline(
    FROZEN_BASELINE_MACRO_F1, color='black', linestyle='--',
    label='frozen seed42'
)
axes[1].set_title('Macro-F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.savefig(output_dir / 'layer4_finetune_seed42_curves.png', dpi=160)
plt.show()

if gain >= REQUIRED_GAIN:
    print('PASS: repeat the same experiment for seeds 123 and 2026.')
else:
    print('NO PASS: keep the frozen encoder and test one augmentation separately.')
print('Do not run the test split here.')

## Как интерпретировать

- `PASS`: partial fine-tuning дал минимум `+0.02` macro-F1 на seed 42. Следующий notebook повторяет строго те же настройки на трёх seed.
- `NO PASS`: мы не сохраняем усложнение как новую основную модель. Следующий отдельный эксперимент проверяет лёгкую spatial/temporal augmentation на frozen baseline.
- Test split в обоих случаях остаётся закрытым до окончательного выбора конфигурации.